# ML

## Grouped, stratified train/test split
All experiments share one split. Complete drillholes stay together; stratification aims for similar Au-class proportions but cannot guarantee them. Of five fixed-seed folds, the one closest to 20% of rows becomes the test set. Selection uses sample count only, never model performance.

Classes are measured-Au training labels, not model predictions: Background < 0.1 ppm; Au-anomalous 0.1 to 0.5 ppm inclusive; High-priority > 0.5 ppm. These are operational screening thresholds, not economic cutoffs. Run data preparation first. No model or preprocessing is fitted here.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

#constants
ROOT = Path.cwd().parent if Path.cwd().name == "ml_pipeline" else Path.cwd()
DATASET_PATH = ROOT / "data" / "extracted_data" / "ml_dataset.csv"
RANDOM_STATE = 67
N_SPLITS = 5
TEST_FRACTION = 0.2
LOW_AU_PPM = 0.1
HIGH_AU_PPM = 0.5
CLASS_NAMES = ["Background", "Au-anomalous", "High-priority"]

In [ ]:
df = pd.read_csv(DATASET_PATH)

#creates labels required for stratification and later classification
au_ppm = df["Au_PPM"]
df["Au_Class"] = "High-priority"
df.loc[au_ppm <= HIGH_AU_PPM, "Au_Class"] = "Au-anomalous"
df.loc[au_ppm < LOW_AU_PPM, "Au_Class"] = "Background"

splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

grouped_splits = list(splitter.split(df, df["Au_Class"], groups=df["CollarId"]))

test_size_differences = [abs(len(test_rows) / len(df) - TEST_FRACTION) for _, test_rows in grouped_split]
selected_fold = test_size_differences.index(min(test_size_differences))
train_indices, test_indices = grouped_splits[selected_fold]

df_train = df.iloc[train_indices].copy()
df_test = df.iloc[test_indices].copy()

train_drillholes = set(df_train["CollarId"])
test_drillholes = set(df_test["CollarId"])

assert train_drillholes.isdisjoint(test_drillholes), "Drillhole leakage"
assert len(df_train) + len(df_test) == len(df), "Incomplete split"

In [ ]:
split_overview = pd.DataFrame(
    
    {
        "rows": [len(df_train), len(df_test)],
        "rows_percent": [len(df_train) / len(df), len(df_test) / len(df)],
        "drillholes": [len(train_drillholes), len(test_drillholes)],
    },

    index=["Train", "Test"],
)

split_overview["rows_percent"] *= 100

class_counts = (
    pd.DataFrame(

        {
            "Train": df_train["Au_Class"].value_counts(),
            "Test": df_test["Au_Class"].value_counts(),
        }

    )

    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)

)

class_percentages = class_counts.div(class_counts.sum()) * 100

print("Split overview:")
print(split_overview.round(1).to_string())
print("\nAu-class counts:")
print(class_counts.to_string())
print("\nAu-class percentages within each split:")
print(class_percentages.round(1).to_string())
print("\nShared drillholes: 0")

## Feature sets for later modelling
Use these explicit columns from the same `df_train` and `df_test` for all three experiments. Metadata stays in the tables for interpretation, but is not a predictor. Fit imputation and models only on training data; do not use the test set to tune them.

In [ ]:
swir_features = ["900D_Fe", "900W_Fe", "1900D_Water", "2200D_AlOH", "2200W_AlOH", "2250D_FeOH", "2250W_FeOH"]
tir_features = ["8625D_Quartz", "11300D_Carbonate", "11300W_Carbonate"]

feature_sets = {

    "SWIR": swir_features,
    "TIR": tir_features,
    "combined": swir_features + tir_features

}

y_train = df_train["Au_Class"]
y_test = df_test["Au_Class"]